# Compare Results: Distillation vs From Scratch

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/marjanstoimcev/DINOv3distill/blob/main/compare_results.ipynb)

This notebook compares the results from:
1. `dinov3_distillation.ipynb` - YOLO11l-seg with DINOv3 distillation pretraining
2. `train_from_scratch.ipynb` - YOLO11l-seg trained from scratch

**Run this after completing both training notebooks.**

In [ ]:
import json
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
from pathlib import Path

# Clean plot style
plt.style.use('default')
plt.rcParams.update({
    'figure.facecolor': 'white',
    'axes.facecolor': 'white',
    'axes.spines.top': False,
    'axes.spines.right': False,
    'axes.spines.left': True,
    'axes.spines.bottom': True,
    'axes.linewidth': 0.8,
    'axes.grid': False,
    'font.size': 11,
    'axes.titlesize': 13,
    'axes.labelsize': 11,
    'xtick.labelsize': 10,
    'ytick.labelsize': 10,
    'legend.fontsize': 10,
    'figure.titlesize': 14,
})

# Color palette
COLORS = {
    'distillation': '#2563eb',  # Blue
    'scratch': '#dc2626',       # Red
    'accent': '#059669',        # Green
}

## 1. Load Metrics

In [ ]:
# Define paths to training outputs
DISTILL_DIR = Path("./output/finetune_seg/yolo11l_seg_taco")
SCRATCH_DIR = Path("./output/from_scratch/yolo11l_seg_taco_scratch")

# Metric files
metrics_files = {
    "DINOv3 Distillation": "./output/finetune_seg/metrics_distillation.json",
    "From Scratch": "./output/from_scratch/metrics_scratch.json"
}

results = {}
for name, path in metrics_files.items():
    try:
        with open(path, 'r') as f:
            results[name] = json.load(f)
        print(f"Loaded: {name}")
    except FileNotFoundError:
        print(f"Not found: {path}")
        print(f"  Run the corresponding notebook first!")

if len(results) < 2:
    print("\nPlease run both training notebooks before comparing!")

In [ ]:
# Display raw metrics
for name, metrics in results.items():
    print(f"\n{'='*50}")
    print(f"{name}")
    print(f"{'='*50}")
    for key, value in metrics.items():
        if key == 'hyperparameters':
            print(f"  {key}:")
            for hk, hv in value.items():
                print(f"    {hk}: {hv}")
        elif isinstance(value, float):
            print(f"  {key}: {value:.4f}")
        else:
            print(f"  {key}: {value}")

## 2. Training Curves

Compare training and validation loss/metrics over epochs.

In [ ]:
def load_training_csv(run_dir):
    """Load training results from Ultralytics CSV output."""
    csv_path = run_dir / "results.csv"
    if csv_path.exists():
        df = pd.read_csv(csv_path)
        # Clean column names (remove leading/trailing spaces)
        df.columns = df.columns.str.strip()
        return df
    return None

# Load training histories
distill_history = load_training_csv(DISTILL_DIR)
scratch_history = load_training_csv(SCRATCH_DIR)

if distill_history is not None:
    print(f"Distillation: {len(distill_history)} epochs recorded")
    print(f"  Columns: {list(distill_history.columns)[:10]}...")
else:
    print("Distillation results.csv not found")

if scratch_history is not None:
    print(f"From Scratch: {len(scratch_history)} epochs recorded")
else:
    print("From Scratch results.csv not found")

In [ ]:
def plot_training_curves(distill_df, scratch_df, metric_cols, titles, ylabel):
    """Plot training curves for multiple metrics."""
    n_metrics = len(metric_cols)
    fig, axes = plt.subplots(1, n_metrics, figsize=(5 * n_metrics, 4))
    
    if n_metrics == 1:
        axes = [axes]
    
    for ax, col, title in zip(axes, metric_cols, titles):
        # Plot distillation
        if distill_df is not None and col in distill_df.columns:
            epochs_d = distill_df['epoch'] if 'epoch' in distill_df.columns else range(1, len(distill_df) + 1)
            ax.plot(epochs_d, distill_df[col], 
                    color=COLORS['distillation'], linewidth=2, 
                    label='DINOv3 Distillation')
        
        # Plot from scratch
        if scratch_df is not None and col in scratch_df.columns:
            epochs_s = scratch_df['epoch'] if 'epoch' in scratch_df.columns else range(1, len(scratch_df) + 1)
            ax.plot(epochs_s, scratch_df[col], 
                    color=COLORS['scratch'], linewidth=2, 
                    label='From Scratch')
        
        ax.set_xlabel('Epoch')
        ax.set_ylabel(ylabel)
        ax.set_title(title, fontweight='bold')
        ax.legend(frameon=False)
        
        # Clean up ticks
        ax.tick_params(direction='out', length=4)
    
    plt.tight_layout()
    return fig

In [ ]:
# Plot loss curves
if distill_history is not None or scratch_history is not None:
    # Common loss columns in Ultralytics output
    loss_cols = ['train/box_loss', 'train/seg_loss', 'train/cls_loss']
    loss_titles = ['Box Loss', 'Segmentation Loss', 'Classification Loss']
    
    # Check which columns exist
    available_cols = []
    available_titles = []
    for col, title in zip(loss_cols, loss_titles):
        if (distill_history is not None and col in distill_history.columns) or \
           (scratch_history is not None and col in scratch_history.columns):
            available_cols.append(col)
            available_titles.append(title)
    
    if available_cols:
        fig = plot_training_curves(distill_history, scratch_history, 
                                   available_cols, available_titles, 'Loss')
        plt.suptitle('Training Loss Curves', fontsize=14, fontweight='bold', y=1.02)
        plt.savefig('training_loss_curves.png', dpi=150, bbox_inches='tight', 
                    facecolor='white', edgecolor='none')
        plt.show()
    else:
        print("Loss columns not found in training history")
else:
    print("No training history available")

In [ ]:
# Plot mAP curves
if distill_history is not None or scratch_history is not None:
    # mAP columns
    map_cols = ['metrics/mAP50(B)', 'metrics/mAP50-95(B)', 
                'metrics/mAP50(M)', 'metrics/mAP50-95(M)']
    map_titles = ['Box mAP50', 'Box mAP50-95', 'Mask mAP50', 'Mask mAP50-95']
    
    # Check which columns exist
    available_cols = []
    available_titles = []
    for col, title in zip(map_cols, map_titles):
        if (distill_history is not None and col in distill_history.columns) or \
           (scratch_history is not None and col in scratch_history.columns):
            available_cols.append(col)
            available_titles.append(title)
    
    if available_cols:
        fig = plot_training_curves(distill_history, scratch_history, 
                                   available_cols, available_titles, 'mAP')
        plt.suptitle('Validation mAP Curves', fontsize=14, fontweight='bold', y=1.02)
        plt.savefig('validation_map_curves.png', dpi=150, bbox_inches='tight',
                    facecolor='white', edgecolor='none')
        plt.show()
    else:
        print("mAP columns not found in training history")
else:
    print("No training history available")

## 3. Final Metrics Comparison

In [ ]:
# Create comparison dataframe
if len(results) >= 2:
    comparison_data = []
    
    for name, metrics in results.items():
        comparison_data.append({
            "Method": name,
            "Box mAP50": metrics.get("box_map50", 0),
            "Box mAP50-95": metrics.get("box_map", 0),
            "Mask mAP50": metrics.get("seg_map50", 0),
            "Mask mAP50-95": metrics.get("seg_map", 0),
        })
    
    df = pd.DataFrame(comparison_data)
    df = df.set_index("Method")
    
    print("\n" + "="*70)
    print("FINAL METRICS COMPARISON")
    print("="*70)
    print(df.to_string())
    print("="*70)

In [ ]:
# Calculate improvement
if len(results) >= 2:
    distill = results.get("DINOv3 Distillation", {})
    scratch = results.get("From Scratch", {})
    
    print("\n" + "="*70)
    print("IMPROVEMENT (Distillation vs From Scratch)")
    print("="*70)
    
    metrics_to_compare = [
        ("Box mAP50", "box_map50"),
        ("Box mAP50-95", "box_map"),
        ("Mask mAP50", "seg_map50"),
        ("Mask mAP50-95", "seg_map"),
    ]
    
    improvements = []
    for display_name, key in metrics_to_compare:
        d_val = distill.get(key, 0)
        s_val = scratch.get(key, 0)
        
        if s_val > 0:
            improvement = ((d_val - s_val) / s_val) * 100
            abs_diff = d_val - s_val
            improvements.append((display_name, abs_diff, improvement))
            symbol = "+" if improvement > 0 else ""
            print(f"  {display_name:15s}: {symbol}{abs_diff:.4f} ({symbol}{improvement:.2f}%)")
        else:
            print(f"  {display_name:15s}: N/A")
    
    print("="*70)

In [ ]:
# Bar chart comparison with clean styling
if len(results) >= 2:
    fig, axes = plt.subplots(1, 2, figsize=(12, 5))
    
    methods = list(results.keys())
    colors = [COLORS['distillation'], COLORS['scratch']]
    
    # Box mAP
    ax = axes[0]
    x = np.arange(2)
    width = 0.35
    
    map50_vals = [results[m].get("box_map50", 0) for m in methods]
    map_vals = [results[m].get("box_map", 0) for m in methods]
    
    bars1 = ax.bar(x - width/2, map50_vals, width, label='mAP50', 
                   color=[COLORS['distillation'], COLORS['scratch']], alpha=0.9)
    bars2 = ax.bar(x + width/2, map_vals, width, label='mAP50-95', 
                   color=[COLORS['distillation'], COLORS['scratch']], alpha=0.5)
    
    ax.set_ylabel('mAP Score')
    ax.set_title('Box Detection', fontweight='bold')
    ax.set_xticks(x)
    ax.set_xticklabels(['Distillation', 'From Scratch'])
    ax.legend(frameon=False, loc='upper right')
    ax.set_ylim(0, max(map50_vals + map_vals) * 1.15)
    
    # Add value labels
    for bar in bars1:
        height = bar.get_height()
        ax.annotate(f'{height:.3f}', xy=(bar.get_x() + bar.get_width()/2, height),
                    xytext=(0, 3), textcoords='offset points', ha='center', fontsize=9)
    for bar in bars2:
        height = bar.get_height()
        ax.annotate(f'{height:.3f}', xy=(bar.get_x() + bar.get_width()/2, height),
                    xytext=(0, 3), textcoords='offset points', ha='center', fontsize=9)
    
    # Mask mAP
    ax = axes[1]
    
    map50_vals = [results[m].get("seg_map50", 0) for m in methods]
    map_vals = [results[m].get("seg_map", 0) for m in methods]
    
    bars1 = ax.bar(x - width/2, map50_vals, width, label='mAP50', 
                   color=[COLORS['distillation'], COLORS['scratch']], alpha=0.9)
    bars2 = ax.bar(x + width/2, map_vals, width, label='mAP50-95', 
                   color=[COLORS['distillation'], COLORS['scratch']], alpha=0.5)
    
    ax.set_ylabel('mAP Score')
    ax.set_title('Instance Segmentation', fontweight='bold')
    ax.set_xticks(x)
    ax.set_xticklabels(['Distillation', 'From Scratch'])
    ax.legend(frameon=False, loc='upper right')
    ax.set_ylim(0, max(map50_vals + map_vals) * 1.15)
    
    # Add value labels
    for bar in bars1:
        height = bar.get_height()
        ax.annotate(f'{height:.3f}', xy=(bar.get_x() + bar.get_width()/2, height),
                    xytext=(0, 3), textcoords='offset points', ha='center', fontsize=9)
    for bar in bars2:
        height = bar.get_height()
        ax.annotate(f'{height:.3f}', xy=(bar.get_x() + bar.get_width()/2, height),
                    xytext=(0, 3), textcoords='offset points', ha='center', fontsize=9)
    
    plt.suptitle('YOLO11l-seg: DINOv3 Distillation vs Training from Scratch', 
                 fontsize=14, fontweight='bold')
    plt.tight_layout()
    plt.savefig('comparison_chart.png', dpi=150, bbox_inches='tight',
                facecolor='white', edgecolor='none')
    plt.show()

## 4. Learning Rate & Loss Analysis

In [ ]:
# Plot learning rate schedule comparison
if distill_history is not None or scratch_history is not None:
    lr_col = 'lr/pg0'  # Common LR column in Ultralytics
    
    has_lr = False
    if distill_history is not None and lr_col in distill_history.columns:
        has_lr = True
    if scratch_history is not None and lr_col in scratch_history.columns:
        has_lr = True
    
    if has_lr:
        fig, ax = plt.subplots(figsize=(8, 4))
        
        if distill_history is not None and lr_col in distill_history.columns:
            epochs_d = range(1, len(distill_history) + 1)
            ax.plot(epochs_d, distill_history[lr_col], 
                    color=COLORS['distillation'], linewidth=2, 
                    label='DINOv3 Distillation')
        
        if scratch_history is not None and lr_col in scratch_history.columns:
            epochs_s = range(1, len(scratch_history) + 1)
            ax.plot(epochs_s, scratch_history[lr_col], 
                    color=COLORS['scratch'], linewidth=2, 
                    label='From Scratch')
        
        ax.set_xlabel('Epoch')
        ax.set_ylabel('Learning Rate')
        ax.set_title('Learning Rate Schedule', fontweight='bold')
        ax.legend(frameon=False)
        ax.tick_params(direction='out', length=4)
        
        plt.tight_layout()
        plt.savefig('learning_rate_schedule.png', dpi=150, bbox_inches='tight',
                    facecolor='white', edgecolor='none')
        plt.show()
    else:
        print("Learning rate column not found in training history")

## 5. Convergence Analysis

In [ ]:
# Analyze convergence speed
def find_convergence_epoch(df, metric_col, threshold_pct=0.95):
    """Find epoch where model reaches threshold % of final performance."""
    if df is None or metric_col not in df.columns:
        return None
    
    values = df[metric_col].values
    final_val = values[-1]
    threshold = final_val * threshold_pct
    
    for i, val in enumerate(values):
        if val >= threshold:
            return i + 1  # 1-indexed epochs
    return len(values)

if distill_history is not None or scratch_history is not None:
    map_col = 'metrics/mAP50-95(M)'  # Mask mAP50-95
    
    print("\n" + "="*70)
    print("CONVERGENCE ANALYSIS")
    print("="*70)
    print("Epochs to reach 95% of final mAP50-95 (Mask):")
    
    conv_distill = find_convergence_epoch(distill_history, map_col)
    conv_scratch = find_convergence_epoch(scratch_history, map_col)
    
    if conv_distill:
        print(f"  DINOv3 Distillation: Epoch {conv_distill}")
    else:
        print(f"  DINOv3 Distillation: N/A")
    
    if conv_scratch:
        print(f"  From Scratch:        Epoch {conv_scratch}")
    else:
        print(f"  From Scratch:        N/A")
    
    if conv_distill and conv_scratch:
        speedup = conv_scratch - conv_distill
        if speedup > 0:
            print(f"\n  Distillation converges {speedup} epochs faster!")
        elif speedup < 0:
            print(f"\n  From scratch converges {-speedup} epochs faster")
        else:
            print(f"\n  Both converge at the same rate")
    
    print("="*70)

## 6. Side-by-Side Inference Comparison

In [ ]:
# Load and display inference images side by side
from PIL import Image

inference_images = {
    "DINOv3 Distillation": "./output/finetune_seg/inference_distillation.png",
    "From Scratch": "./output/from_scratch/inference_scratch.png"
}

fig, axes = plt.subplots(2, 1, figsize=(18, 10))

for ax, (name, path) in zip(axes, inference_images.items()):
    try:
        img = Image.open(path)
        ax.imshow(img)
        ax.set_title(name, fontsize=13, fontweight='bold', pad=10)
        ax.axis('off')
    except FileNotFoundError:
        ax.text(0.5, 0.5, f"Image not found:\n{path}", 
                ha='center', va='center', fontsize=12,
                transform=ax.transAxes)
        ax.set_title(name, fontsize=13, fontweight='bold')
        ax.axis('off')

plt.suptitle('Inference Comparison', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('inference_comparison.png', dpi=150, bbox_inches='tight',
            facecolor='white', edgecolor='none')
plt.show()

## 7. Summary

In [ ]:
if len(results) >= 2:
    distill = results.get("DINOv3 Distillation", {})
    scratch = results.get("From Scratch", {})
    
    seg_d = distill.get("seg_map", 0)
    seg_s = scratch.get("seg_map", 1)
    seg_improvement = ((seg_d - seg_s) / seg_s) * 100 if seg_s > 0 else 0
    
    print("\n" + "="*70)
    print("SUMMARY")
    print("="*70)
    print(f"\nDataset: TACO (1,499 images, 59 classes)")
    print(f"Model: YOLO11l-seg (27.7M params)")
    
    # Hyperparameters comparison
    print(f"\nDINOv3 Distillation:")
    if 'hyperparameters' in distill:
        hp = distill['hyperparameters']
        print(f"  - Distillation epochs: {hp.get('distill_epochs', 'N/A')}")
        print(f"  - Distillation batch:  {hp.get('distill_batch_size', 'N/A')}")
        print(f"  - Fine-tune epochs:    {hp.get('finetune_epochs', 'N/A')}")
        print(f"  - Fine-tune batch:     {hp.get('finetune_batch_size', 'N/A')}")
        print(f"  - Fine-tune LR:        {hp.get('finetune_lr', 'N/A')}")
    print(f"  - Mask mAP50-95:       {seg_d:.4f}")
    
    print(f"\nFrom Scratch:")
    if 'hyperparameters' in scratch:
        hp = scratch['hyperparameters']
        print(f"  - Train epochs:        {hp.get('epochs', 'N/A')}")
        print(f"  - Batch size:          {hp.get('batch_size', 'N/A')}")
        print(f"  - Learning rate:       {hp.get('lr', 'N/A')}")
    print(f"  - Mask mAP50-95:       {seg_s:.4f}")
    
    print(f"\n{'=' * 40}")
    if seg_improvement > 0:
        print(f"Segmentation Improvement: +{seg_improvement:.2f}%")
        print(f"\nDINOv3 distillation pretraining improved performance!")
    elif seg_improvement < 0:
        print(f"Segmentation Change: {seg_improvement:.2f}%")
        print(f"\nConsider increasing distillation epochs or adjusting hyperparameters.")
    else:
        print(f"No significant difference in segmentation performance.")
    print("="*70)